# AccentSense — UK Regional Accent Classifier
**WavLM Base+ · Attentive Statistics Pooling · 7 UK Dialect Classes**

VIT Bhopal University · AI/ML Capstone Research

---
**Classes:** `RP` · `Scottish` · `Welsh` · `Northern` · `West_Midlands` · `Cockney` · `Irish`

**Hardware:** Colab T4 GPU (Runtime → Change runtime type → T4)

**Steps:**
1. GPU check
2. Install dependencies
3. Clone repo
4. Download VCTK corpus (real UK speakers, 16 kHz)
5. (Optional) Mozilla Common Voice Irish supplement
6. Build speaker-disjoint manifests
7. Train WavLM (20 epochs, cosine LR, class-weighted CE)
8. Evaluate — classification report + confusion matrix
9. Download checkpoint → place in `Backend/checkpoints/`


In [ ]:
# Step 1: Verify GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print('VRAM: %.1f GB' % vram)
else:
    print('WARNING: No GPU. Go to Runtime -> Change runtime type -> T4 GPU')


CUDA: True
GPU: Tesla T4
VRAM: 15.6 GB


In [9]:
# Step 2: Install dependencies
!pip install -q transformers accelerate torchaudio datasets soundfile jiwer scikit-learn pandas
print('All dependencies installed.')


All dependencies installed.


In [3]:
# Step 3: Clone AccentSense repository
import os
# UPDATE: replace with your actual GitHub repo
REPO = 'SameerGera/Accent-Sense'
!git clone https://github.com/{REPO}.git AccentSense
os.chdir('AccentSense/Backend')
import sys
sys.path.insert(0, '.')
print('Working dir:', os.getcwd())


Cloning into 'AccentSense'...
remote: Enumerating objects: 233, done.
remote: Counting objects: 100% (233/233), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 233 (delta 100), reused 216 (delta 83), pack-reused 0 (from 0)
Receiving objects: 100% (233/233), 166.39 KiB | 1.65 MiB/s, done.
Resolving deltas: 100% (100/100), done.
Working dir: /content/AccentSense/Backend


In [4]:
# Step 4: Download VCTK Corpus (University of Edinburgh, free)
# Strategy: curl -L (with browser user-agent) -> wget -> Hugging Face fallback in Step 6
import os, glob

VCTK_URL = 'https://datashare.is.ed.ac.uk/bitstream/handle/10283/3443/VCTK-Corpus-0.92.zip'
USE_HF_VCTK = False
zip_path = 'data/vctk.zip'
os.makedirs('data/vctk', exist_ok=True)

downloaded = False

# Attempt 1: curl -L with browser User-Agent (prevents 403 on Edinburgh DataShare)
print('Attempting download via curl (University of Edinburgh mirror)...')
ret = os.system(f'curl -L -A "Mozilla/5.0 (Windows NT 10.0; Win64; x64)" --retry 2 --retry-delay 3 --max-time 600 -o {zip_path} "{VCTK_URL}"')
if ret == 0 and os.path.exists(zip_path):
    sz = os.path.getsize(zip_path)
    if sz > 1_000_000_000:
        downloaded = True
        print(f'✓ Downloaded via curl: {sz / (1024**3):.2f} GB')
    else:
        print(f'curl download returned small file ({sz} bytes), likely a rate-limit or redirect page.')
        try:
            os.remove(zip_path)
        except OSError:
            pass

# Attempt 2: wget with user-agent
if not downloaded:
    print('Attempting download via wget...')
    ret = os.system(f'wget --user-agent="Mozilla/5.0" --tries=2 --timeout=60 -q -O {zip_path} "{VCTK_URL}"')
    if ret == 0 and os.path.exists(zip_path):
        sz = os.path.getsize(zip_path)
        if sz > 1_000_000_000:
            downloaded = True
            print(f'✓ Downloaded via wget: {sz / (1024**3):.2f} GB')
        else:
            print(f'wget download too small ({sz} bytes).')
            try:
                os.remove(zip_path)
            except OSError:
                pass

# Extract if downloaded, otherwise fall back to Hugging Face
if downloaded:
    print('Extracting VCTK (this may take a few minutes)...')
    ret = os.system(f'cd data && unzip -q -o vctk.zip -d vctk/')
    if ret == 0:
        try:
            os.remove(zip_path)
        except OSError:
            pass
    info = glob.glob('data/vctk/**/speaker-info.txt', recursive=True)
    if info:
        print('✓ Local VCTK extracted. speaker-info.txt:', info[0])
    else:
        print('⚠ speaker-info.txt not found after extraction. Will fall back to Hugging Face.')
        USE_HF_VCTK = True
else:
    print('\n⚠ Edinburgh DataShare server unavailable or download timed out.')
    print('✓ Automatically falling back to Hugging Face in Step 6 (no manual action needed).')
    USE_HF_VCTK = True
    if os.path.exists(zip_path):
        try:
            os.remove(zip_path)
        except OSError:
            pass

if USE_HF_VCTK:
    print('VCTK audio will be streamed & loaded directly from Hugging Face in Step 6.')


Attempting download via curl...
Downloaded via curl: 10.94 GB
Extracting VCTK (this may take a few minutes)...
speaker-info.txt: ['data/vctk/speaker-info.txt']


In [5]:
# Step 5 (Optional): Add Irish English speech supplement
# Supplements VCTK which has very few Irish speakers.
# Priority:
#   1. Public British & Irish Dialects (ylacombe/english_dialects -> irish_male / irish_female) [No Auth Needed]
#   2. Mozilla Common Voice (requires HF_TOKEN if gated)
import os
import soundfile as sf
import numpy as np

cv_extras = []

try:
    from datasets import load_dataset
    irish = []

    # 1. Try open-access English Dialects dataset (public, no auth or gating required)
    print('Trying open English Dialects dataset (no token required)...')
    for subset in ['irish_male', 'irish_female']:
        try:
            ds = load_dataset('ylacombe/english_dialects', subset, split='train', streaming=True)
            for s in ds:
                irish.append({
                    'audio': s['audio'],
                    'speaker': s.get('speaker_id', f'spk_{subset}')
                })
                if len(irish) >= 500:
                    break
            print(f'  ✓ Found {len(irish)} samples from {subset}')
            if len(irish) >= 500:
                break
        except Exception as e:
            print(f'  {subset} not available: {e}')

    # 2. Fallback to Mozilla Common Voice if needed (supports HF_TOKEN)
    if len(irish) < 50:
        print('Trying Mozilla Common Voice...')
        hf_token = os.environ.get('HF_TOKEN')
        if not hf_token:
            try:
                from google.colab import userdata
                hf_token = userdata.get('HF_TOKEN')
            except Exception:
                pass

        IRISH_TAGS = {'Ireland', 'Irish', 'Dublin', 'Northern Ireland'}
        for cv_version in ['mozilla-foundation/common_voice_17_0', 'mozilla-foundation/common_voice_13_0']:
            try:
                token_status = "provided" if hf_token else "none"
                print(f'Trying {cv_version} (token={token_status})...')
                cv = load_dataset(cv_version, 'en', split='train', streaming=True, token=hf_token)
                for s in cv.take(100000):
                    if s.get('accent', '') in IRISH_TAGS:
                        irish.append({
                            'audio': s['audio'],
                            'speaker': s.get('client_id', 'cv_unknown')
                        })
                    if len(irish) >= 500:
                        break
                if irish:
                    break
            except Exception as ve:
                print(f'  {cv_version} failed: {ve}')
                continue

    if irish:
        print(f'\nTotal Irish clips found: {len(irish)}')
        os.makedirs('data/cv_irish', exist_ok=True)
        for i, item in enumerate(irish):
            p = 'data/cv_irish/irish_%04d.wav' % i
            sf.write(p, np.array(item['audio']['array'], dtype=np.float32), item['audio']['sampling_rate'])
            cv_extras.append({
                'path': p,
                'label': 'Irish',
                'speaker': str(item['speaker']),
                'class_idx': 6
            })
        print(f'Saved {len(cv_extras)} Irish WAV files.')
    else:
        print('No Irish clips could be fetched. Continuing with VCTK-only data.')

except Exception as e:
    print(f'Irish supplement error: {e}')
    print('Continuing with VCTK-only data.')


Trying mozilla-foundation/common_voice_13_0...


README.md:   0%|          | 0.00/357 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


  mozilla-foundation/common_voice_13_0 failed: The directory at hf://datasets/mozilla-foundation/common_voice_13_0@ff2bbb54dcdb597100fe534a1b911ff9103f9e22 doesn't contain any data files
Trying mozilla-foundation/common_voice_17_0...


README.md:   0%|          | 0.00/414 [00:00<?, ?B/s]

  mozilla-foundation/common_voice_17_0 failed: The directory at hf://datasets/mozilla-foundation/common_voice_17_0@11dc88355e899d1bf2df74f01b904a8544a17b33 doesn't contain any data files
No Irish clips available from Common Voice. Continuing with VCTK-only.


In [14]:
# Step 6: Build speaker-disjoint manifests (Supports Local VCTK + Hugging Face Fallback)
from collections import Counter
import glob, os, sys
import numpy as np
import soundfile as sf

CLASSES = ('RP', 'Scottish', 'Welsh', 'Northern', 'West_Midlands', 'Cockney', 'Irish')

# Check if local VCTK is present and valid
INFO_FILES = glob.glob('data/vctk/**/speaker-info.txt', recursive=True)
use_local = (not globals().get('USE_HF_VCTK', False)) and len(INFO_FILES) > 0

manifest = []
speaker_counts = Counter()

# ---------------------------------------------------------------------------
# PATH A: Local VCTK extraction available
# ---------------------------------------------------------------------------
if use_local:
    print("=== Building manifest from local VCTK ===")
    INFO = INFO_FILES[0]
    print(f'Speaker info: {INFO}')

    WAVD = None
    for pattern in ['data/vctk/wav48_silence_trimmed', 'data/vctk/wav48', 'data/vctk/wav16']:
        if os.path.isdir(pattern):
            WAVD = pattern
            print(f'✓ Found audio directory: {WAVD}')
            break

    if not WAVD:
        print("Could not find wav directory under data/vctk/. Falling back to Hugging Face...")
        use_local = False

if use_local:
    speaker_accents = {}
    with open(INFO, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('ID'):
                continue
            parts = line.split()
            if len(parts) >= 4:
                speaker_id = parts[0]
                accent = parts[3]
                speaker_accents[speaker_id] = accent

    print(f"Loaded {len(speaker_accents)} speakers from speaker-info.txt")

    ACCENT_TO_CLASS = {
        'RP': 'RP',
        'Scottish': 'Scottish',
        'Welsh': 'Welsh',
        'Northern': 'Northern',
        'West_Midlands': 'West_Midlands',
        'Cockney': 'Cockney',
    }

    print(f"Scanning {WAVD} for audio files...")
    for speaker_dir in sorted(os.listdir(WAVD)):
        speaker_path = os.path.join(WAVD, speaker_dir)
        if not os.path.isdir(speaker_path):
            continue

        speaker_id = speaker_dir
        if speaker_id not in speaker_accents:
            if speaker_id.startswith('p') or speaker_id.startswith('s'):
                lookup_id = speaker_id
            else:
                continue
        else:
            lookup_id = speaker_id

        if lookup_id not in speaker_accents:
            continue

        accent = speaker_accents[lookup_id]
        if accent not in ACCENT_TO_CLASS:
            continue

        class_label = ACCENT_TO_CLASS[accent]
        class_idx = CLASSES.index(class_label)

        audio_files = []
        for ext in ['*.wav', '*.WAV', '*.flac', '*.FLAC']:
            audio_files.extend(glob.glob(os.path.join(speaker_path, ext)))

        for audio_path in audio_files:
            manifest.append({
                'path': audio_path,
                'label': class_label,
                'speaker': speaker_id,
                'class_idx': class_idx
            })
            speaker_counts[class_label] += 1

# ---------------------------------------------------------------------------
# PATH B: Local VCTK not available -> Fall back to Hugging Face datasets
# ---------------------------------------------------------------------------
if not use_local or len(manifest) == 0:
    print("\n=== Local VCTK not found. Streaming & building dataset from Hugging Face ===")
    from datasets import load_dataset
    from src.data.vctk_dataset import VCTK_ACCENT_MAP

    os.makedirs('data/vctk_hf', exist_ok=True)
    loaded_from_hf = False

    # Attempt 1: CSTR-Edinburgh/vctk on Hugging Face
    try:
        print("Connecting to 'CSTR-Edinburgh/vctk' (mic1, streaming)...")
        ds = load_dataset("CSTR-Edinburgh/vctk", "mic1", split="train", streaming=True)
        class_samples = Counter()
        MAX_PER_CLASS = 500

        for idx, row in enumerate(ds):
            accent_tag = str(row.get("accent", row.get("region", "")))
            cls = VCTK_ACCENT_MAP.get(accent_tag)
            if not cls:
                for k, v in VCTK_ACCENT_MAP.items():
                    if k.lower() in accent_tag.lower():
                        cls = v
                        break
            if not cls or cls not in CLASSES or cls == 'Irish':
                continue

            if class_samples[cls] >= MAX_PER_CLASS:
                continue

            spk = str(row.get("speaker_id", f"spk_{cls}"))
            audio = row["audio"]
            p = f"data/vctk_hf/{spk}_{idx:05d}.wav"
            if not os.path.exists(p):
                sf.write(p, np.array(audio["array"], dtype=np.float32), audio["sampling_rate"])

            manifest.append({
                'path': p,
                'label': cls,
                'speaker': spk,
                'class_idx': CLASSES.index(cls)
            })
            speaker_counts[cls] += 1
            class_samples[cls] += 1

            if len(manifest) % 150 == 0:
                print(f"  Downloaded {len(manifest)} samples from HF VCTK...")

            if all(class_samples[c] >= MAX_PER_CLASS for c in ['RP', 'Scottish', 'Welsh', 'Northern', 'West_Midlands']):
                break

        if len(manifest) > 100:
            loaded_from_hf = True
            print(f"✓ Built manifest with {len(manifest)} clips from CSTR-Edinburgh/vctk")
    except Exception as e:
        print(f"  CSTR-Edinburgh/vctk stream issue: {e}")

    # Attempt 2: ylacombe/english_dialects (fast open dialect dataset covering UK accents)
    if not loaded_from_hf or len(manifest) < 100:
        print("\nLoading from 'ylacombe/english_dialects' (open UK dialect corpus)...")
        DIALECT_MAP = {
            'southern_male': 'RP',
            'southern_female': 'RP',
            'scottish_male': 'Scottish',
            'scottish_female': 'Scottish',
            'welsh_male': 'Welsh',
            'welsh_female': 'Welsh',
            'northern_male': 'Northern',
            'northern_female': 'Northern',
            'midlands_male': 'West_Midlands',
            'midlands_female': 'West_Midlands',
        }
        for subset, cls in DIALECT_MAP.items():
            try:
                ds = load_dataset('ylacombe/english_dialects', subset, split='train', streaming=True)
                cnt = 0
                for idx, row in enumerate(ds):
                    spk = str(row.get("speaker_id", f"spk_{subset}"))
                    audio = row["audio"]
                    p = f"data/vctk_hf/{subset}_{idx:04d}.wav"
                    if not os.path.exists(p):
                        sf.write(p, np.array(audio["array"], dtype=np.float32), audio["sampling_rate"])

                    manifest.append({
                        'path': p,
                        'label': cls,
                        'speaker': spk,
                        'class_idx': CLASSES.index(cls)
                    })
                    speaker_counts[cls] += 1
                    cnt += 1
                    if cnt >= 250:
                        break
                print(f"  ✓ {subset} -> {cls}: {cnt} samples")
            except Exception as se:
                print(f"  Could not load {subset}: {se}")

# ---------------------------------------------------------------------------
# Add Irish English Supplement (from Step 5)
# ---------------------------------------------------------------------------
if 'cv_extras' in globals() and cv_extras:
    existing_irish_paths = {m['path'] for m in manifest if m['label'] == 'Irish'}
    new_cv = [c for c in cv_extras if c['path'] not in existing_irish_paths]
    manifest.extend(new_cv)
    print(f'\n+ Added {len(new_cv)} Irish samples from Step 5')
    for item in new_cv:
        speaker_counts['Irish'] += 1

print(f"\n✓ Total audio files in manifest: {len(manifest)}")
print("\nClass distribution:")
for cls in CLASSES:
    count = speaker_counts[cls]
    print(f'  {cls:<15}: {count:5d}')

# ---------------------------------------------------------------------------
# Create speaker-disjoint splits and save
# ---------------------------------------------------------------------------
print("\nCreating speaker-disjoint train/val/test splits...")
from src.data.vctk_dataset import speaker_disjoint_splits, save_manifest

train_m, val_m, test_m = speaker_disjoint_splits(manifest)
print(f'Splits → Train: {len(train_m)} | Val: {len(val_m)} | Test: {len(test_m)}')

os.makedirs('data/manifests', exist_ok=True)
save_manifest(train_m, 'data/manifests/train.json')
save_manifest(val_m,   'data/manifests/val.json')
save_manifest(test_m,  'data/manifests/test.json')

print('✓ Manifests saved to data/manifests/')


=== Building manifest from local VCTK ===
Speaker info: data/vctk/speaker-info.txt
✓ Found audio directory: data/vctk/wav48_silence_trimmed
Parsing speaker info...
Loaded 110 speakers
Scanning data/vctk/wav48_silence_trimmed for audio files...
  ⚠ p225: accent 'English' not in class mapping, skipping
  ⚠ p226: accent 'English' not in class mapping, skipping
  ⚠ p227: accent 'English' not in class mapping, skipping
  ⚠ p228: accent 'English' not in class mapping, skipping
  ⚠ p229: accent 'English' not in class mapping, skipping
  ⚠ p230: accent 'English' not in class mapping, skipping
  ⚠ p231: accent 'English' not in class mapping, skipping
  ⚠ p232: accent 'English' not in class mapping, skipping
  ⚠ p233: accent 'English' not in class mapping, skipping
  ⚠ p236: accent 'English' not in class mapping, skipping
  ⚠ p238: accent 'NorthernIrish' not in class mapping, skipping
  ⚠ p239: accent 'English' not in class mapping, skipping
  ⚠ p240: accent 'English' not in class mapping, skipp

In [ ]:
## Step 7: Training (7-class model: RP, Scottish, Welsh, Northern, West_Midlands, Cockney, Irish)
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, balanced_accuracy_score
from src.data.vctk_dataset import UKAccentDataset, collate_pad, load_manifest
from src.models.wavlm_classifier import WavLMAccentClassifier
from collections import Counter
import torch, os

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASSES = ('RP', 'Scottish', 'Welsh', 'Northern', 'West_Midlands', 'Cockney', 'Irish')
N       = len(CLASSES)  # 7
EPOCHS  = 20
BATCH   = 8
LR      = 3e-4

print(f"Training {N}-class model: {CLASSES}")
print(f"Device: {DEVICE}\n")

# Load model
model = WavLMAccentClassifier(
    pretrained_model_name='microsoft/wavlm-base-plus',
    num_classes=N,
    freeze_encoder=True,
    unfreeze_top_k_layers=2,
    dropout_p=0.3,
).to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Trainable: %d / %d (%.1f%%)\n' % (trainable, total, 100*trainable/total))

# Load datasets
tr_ds = UKAccentDataset(load_manifest('data/manifests/train.json'), augment=True)
va_ds = UKAccentDataset(load_manifest('data/manifests/val.json'),   augment=False)
tr_dl = DataLoader(tr_ds, batch_size=BATCH, shuffle=True,  collate_fn=collate_pad, num_workers=2, pin_memory=True)
va_dl = DataLoader(va_ds, batch_size=BATCH, shuffle=False, collate_fn=collate_pad, num_workers=2)

print(f"Train: {len(tr_ds)} samples ({len(tr_dl)} batches)")
print(f"Val:   {len(va_ds)} samples ({len(va_dl)} batches)\n")

# Class-weighted loss to handle imbalance
cnt = Counter(e['class_idx'] for e in load_manifest('data/manifests/train.json'))
w   = torch.tensor([1.0 / cnt.get(i, 1) for i in range(N)], dtype=torch.float32)
w   = w / w.sum() * N
criterion = torch.nn.CrossEntropyLoss(weight=w.to(DEVICE))

print("Class distribution in training set:")
for i, cls in enumerate(CLASSES):
    count = cnt.get(i, 0)
    print(f"  {cls:<12}: {count:6d} (weight: {w[i]:.3f})")
print()

# Optimizer & scheduler
optimizer  = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
T          = len(tr_dl) * EPOCHS
scheduler  = get_cosine_schedule_with_warmup(optimizer, T // 10, T)

best = 0.0
os.makedirs('checkpoints', exist_ok=True)

print(f"{'Ep':<4} {'Loss':<10} {'TrF1':<10} {'ValF1':<10} {'BalAcc':<10} Status")
print("-" * 60)

for ep in range(1, EPOCHS + 1):
    # Training loop
    model.train()
    el, pp, ll = 0.0, [], []
    for wav, lb, mask in tr_dl:
        wav, lb, mask = wav.to(DEVICE), lb.to(DEVICE), mask.to(DEVICE)
        optimizer.zero_grad()
        out  = model(wav, attention_mask=mask)
        loss = criterion(out['logits'], lb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        el += loss.item()
        pp.extend(out['logits'].argmax(-1).cpu().tolist())
        ll.extend(lb.cpu().tolist())

    tr_f1 = f1_score(ll, pp, average='macro', zero_division=0)

    # Validation loop
    model.eval()
    vp, vl = [], []
    with torch.no_grad():
        for wav, lb, mask in va_dl:
            out = model(wav.to(DEVICE), attention_mask=mask.to(DEVICE))
            vp.extend(out['logits'].argmax(-1).cpu().tolist())
            vl.extend(lb.tolist())

    val_f1 = f1_score(vl, vp, average='macro', zero_division=0)
    bal_acc = balanced_accuracy_score(vl, vp)

    status = ""
    if val_f1 > best:
        best = val_f1
        torch.save(model.state_dict(), 'checkpoints/best_wavlm_accentsense.pt')
        status = "✓ BEST"

    print(f'{ep:<4} {el/len(tr_dl):<10.4f} {tr_f1:<10.4f} {val_f1:<10.4f} {bal_acc:<10.4f} {status}')

print("-" * 60)
print(f'✓ Training complete.')
print(f'  Best Val Macro-F1: {best:.4f}')
print(f'  Checkpoint: checkpoints/best_wavlm_accentsense.pt')


Training 4-class model: ('RP', 'Scottish', 'Welsh', 'Irish')
Device: cuda



config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  378MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

model.safetensors: downloading bytes:           |  0.00B            

Trainable: 14672941 / 94878069 (15.5%)

Train: 12266 samples (1534 batches)
Val:   1564 samples (196 batches)

Class distribution in training set:
  RP          :      0 (weight: 1.999)
  Scottish    :  11516 (weight: 0.000)
  Welsh       :    750 (weight: 0.003)
  Irish       :      0 (weight: 1.999)

Ep   Loss       TrF1       ValF1      BalAcc     Status
------------------------------------------------------------


In [ ]:
# Step 8: Test-set evaluation
from sklearn.metrics import classification_report, confusion_matrix

model.load_state_dict(torch.load('checkpoints/best_wavlm_accentsense.pt', map_location=DEVICE))
model.eval()

te_ds = UKAccentDataset(load_manifest('data/manifests/test.json'), augment=False)
te_dl = DataLoader(te_ds, batch_size=BATCH, shuffle=False, collate_fn=collate_pad)

tp, tl = [], []
with torch.no_grad():
    for wav, lb, mask in te_dl:
        out = model(wav.to(DEVICE), attention_mask=mask.to(DEVICE))
        tp.extend(out['logits'].argmax(-1).cpu().tolist())
        tl.extend(lb.tolist())

print('=' * 60)
print('TEST SET RESULTS')
print('=' * 60)
print(classification_report(tl, tp, target_names=CLASSES))
print('Confusion Matrix:')
print(confusion_matrix(tl, tp))
test_f1 = f1_score(tl, tp, average='macro', zero_division=0)
print('Final Test Macro-F1: %.4f' % test_f1)


In [ ]:
# Step 9: Download checkpoint
from google.colab import files
ckpt = 'checkpoints/best_wavlm_accentsense.pt'
size = os.path.getsize(ckpt) / 1024 / 1024
print('Downloading checkpoint (%.1f MB)...' % size)
files.download(ckpt)
print('Place at: Backend/checkpoints/best_wavlm_accentsense.pt')
print('Restart uvicorn -> GET /health shows model_loaded: true')


## After Training

1. Download `best_wavlm_accentsense.pt` from Step 9
2. Copy to `Backend/checkpoints/best_wavlm_accentsense.pt`
3. Restart backend: `uvicorn src.api.main:app --reload`
4. `GET /health` → `model_loaded: true`, `supported_classes: [RP, Scottish, Welsh, Northern, West_Midlands, Cockney, Irish]`

**Expected Test Macro-F1:** >0.70 on VCTK real recordings

> **Cockney tip:** VCTK has few London speakers. If Cockney F1 is low (<0.5),
> add more Common Voice clips tagged 'London' in Step 5 and re-run splits.
